Этот файл содержит исходные ячейки и сохранённые результаты AlphaFold2 из общей Colab-сессии. При подготовке отчёта ячейки отделены по предсказателям. Сохранены реальные ошибки настройки и итоговый успешный запуск. Полный исходный экспорт общей сессии сохранён отдельно в рабочем архиве.


In [1]:
#@title Проверка сохранённых файлов и GPU
from pathlib import Path
import os, sys, subprocess
print(sys.version)
print('Files:',[p.name for p in Path('/content').iterdir()])
print('LD_LIBRARY_PATH:',os.environ.get('LD_LIBRARY_PATH'))
print('Driver directories:',[(str(p),p.exists()) for p in [Path('/usr/local/nvidia/lib'),Path('/usr/local/nvidia/lib64')]])
p=subprocess.run(['nvidia-smi'],capture_output=True,text=True)
print(p.stdout,p.stderr)


3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
Files: ['.config', 'sample_data']
LD_LIBRARY_PATH: /usr/lib64-nvidia
Driver directories: [('/usr/local/nvidia/lib', False), ('/usr/local/nvidia/lib64', False)]
Wed Nov 18 08:30:09 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   44C    P8    

In [2]:
#@title AlphaFold2 isolated environment with NVIDIA driver path
import os, sys, subprocess
def run_live(command, **kwargs):
    print(command,flush=True)
    p=subprocess.Popen(command,shell=True,text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,**kwargs)
    for line in p.stdout: print(line,end='',flush=True)
    if p.wait(): raise RuntimeError('Command failed')
run_live(f'{sys.executable} -m pip install -q virtualenv')
run_live(f'{sys.executable} -m virtualenv /content/af2-env')
af_env=os.environ.copy()
af_env.pop('PYTHONPATH',None)
assert os.path.isdir('/usr/lib64-nvidia')
af_env['LD_LIBRARY_PATH']='/usr/lib64-nvidia'
run_live("/content/af2-env/bin/python -m pip install 'colabfold[alphafold] @ git+https://github.com/sokrypton/ColabFold' 'jax[cuda12]>=0.6.2,<0.12'",env=af_env)


/usr/bin/python3 -m pip install -q virtualenv
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.7/5.7 MB 68.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.1/472.1 kB 22.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 222.4/222.4 kB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.0/130.0 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.5/98.5 kB 9.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-core 1.2.7 requires packaging<26.0.0,>=23.2.0, but you have packaging 26.3 which is incompatible.
/usr/bin/python3 -m virtualenv /content/af2-env
created virtual environment CPython3.12.12.final.0-64-x86_64 in 635ms
  creator CPython3Posix(dest=/content/af2-env, clear=False, no_vcs_ignore=False, venv_redirect=None, global=False)
  seeder FromAppData(download=False, pip=

In [3]:
#@title AlphaFold2: GPU test and five predictions for row 29
from pathlib import Path
import shlex
af_dir=Path('/content/af2-run')
af_dir.mkdir(exist_ok=True)
(af_dir/'row29.fasta').write_text('>assignment5_row29_af2\nMASVQQGEKQLFEKFWRGTFKAVATPRPESIIVASITARKPLPRTEPQNNPVVPAQDGPSEKLGQHLATEPLGTNSWERDKTCRELGATRGHSASHDKDL\n')
probe="import jax, jax.numpy as j; print(jax.__version__,jax.devices()); assert any(d.platform=='gpu' for d in jax.devices()); print(j.dot(j.ones((32,32)),j.ones((32,32))).block_until_ready()[0,0]); import colabfold.batch; print('AF2 imports OK')"
run_live('/content/af2-env/bin/python -c '+shlex.quote(probe),cwd=str(af_dir),env=af_env)
run_live('/content/af2-env/bin/python -m pip freeze > dependencies.txt',cwd=str(af_dir),env=af_env)
run_live('/content/af2-env/bin/colabfold_batch /content/af2-run/row29.fasta /content/af2-run/results --model-type alphafold2_ptm --num-models 5 --num-recycle 3 --num-relax 0 --msa-mode mmseqs2_uniref_env',cwd=str(af_dir),env=af_env)


/content/af2-env/bin/python -c 'import jax, jax.numpy as j; print(jax.__version__,jax.devices()); assert any(d.platform=='"'"'gpu'"'"' for d in jax.devices()); print(j.dot(j.ones((32,32)),j.ones((32,32))).block_until_ready()[0,0]); import colabfold.batch; print('"'"'AF2 imports OK'"'"')'
0.11.2 [CudaDevice(id=0)]
32.0
AF2 imports OK
/content/af2-env/bin/python -m pip freeze > dependencies.txt
/content/af2-env/bin/colabfold_batch /content/af2-run/row29.fasta /content/af2-run/results --model-type alphafold2_ptm --num-models 5 --num-recycle 3 --num-relax 0 --msa-mode mmseqs2_uniref_env
2026-11-18 08:35:15,227 Running colabfold 1.6.3 (eb061aa6706959fdffe15e86d27ca08083d6ed8d)


limited shared resource only capable of processing a few thousand MSAs per day. Please
submit jobs only from a single IP address. We reserve the right to limit access to the
server case-by-case when usage exceeds fair use. If you require more MSAs: You can 
precompute all MSAs with `colabfold_search` or host your ow

In [5]:
#@title Run AlphaFold2 with headless plotting backend
af_env['MPLBACKEND']='Agg'
run_live('/content/af2-env/bin/colabfold_batch /content/af2-run/row29.fasta /content/af2-run/results --model-type alphafold2_ptm --num-models 5 --num-recycle 3 --num-relax 0 --msa-mode mmseqs2_uniref_env',cwd='/content/af2-run',env=af_env)
assert len(list(Path('/content/af2-run/results').glob('*unrelaxed_rank_*.pdb')))==5
print('All five AlphaFold2 PDB files generated')


/content/af2-env/bin/colabfold_batch /content/af2-run/row29.fasta /content/af2-run/results --model-type alphafold2_ptm --num-models 5 --num-recycle 3 --num-relax 0 --msa-mode mmseqs2_uniref_env

limited shared resource only capable of processing a few thousand MSAs per day. Please
submit jobs only from a single IP address. We reserve the right to limit access to the
server case-by-case when usage exceeds fair use. If you require more MSAs: You can 
precompute all MSAs with `colabfold_search` or host your own API and pass it to `--host-url`

2026-11-18 08:36:47,028 Running colabfold 1.6.3 (eb061aa6706959fdffe15e86d27ca08083d6ed8d)
2026-11-18 08:36:47,487 Running on GPU
2026-11-18 08:36:47,617 Found 5 citations for tools or databases
2026-11-18 08:36:47,617 Query 1/1: assignment5_row29_af2 (length 100)
2026-11-18 08:36:48,432 generated new fontManager

COMPLETE: 100%|██████████| 150/150 [elapsed: 00:00 remaining: 00:00]
W1118 08:37:07.633052    2213 cuda_timer.cc:88] Delay kernel timed o

## Сохранённая структура
Оригинальный PDB, полученный при выполнении расчёта, вложен в этот файл при подготовке комплекта. Его отдельная копия находится в predictions/.
<a download="alphafold2_rank1.pdb" href="...">Сохранить PDB alphafold2_rank1.pdb</a>
